# RTL-SDR as a Spectrum Analyzer

This notebook turns either a local IQ sweep capture or a synthetic wideband signal into a spectrum-viewing workflow. Live SDR use is optional and explicitly gated by `probe_rtlsdr()`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
CAPTURE_ROOT = ROOT / "assets" / "local"
capture_status = probe_rtlsdr()
display(Markdown(
    f"**RTL-SDR status:** installed={capture_status['installed']}, "
    f"available={capture_status['available']}. {capture_status['message']}"
))


In [ ]:
capture_path = CAPTURE_ROOT / "spectrum_sweep_iq.npz"
if capture_path.exists():
    fs_iq, iq = load_complex_capture(capture_path)
    print(f"Loaded local capture: {capture_path.name}, fs={fs_iq}")
else:
    fs_iq = 240_000
    t = np.arange(0, 1.0, 1 / fs_iq)
    iq = (
        1.0 * np.exp(1j * 2 * np.pi * 20_000 * t)
        + 0.5 * np.exp(1j * 2 * np.pi * -40_000 * t)
        + 0.25 * np.exp(1j * 2 * np.pi * 70_000 * t)
    )
    iq += 0.03 * (np.random.default_rng(42).normal(size=len(t)) + 1j * np.random.default_rng(43).normal(size=len(t)))
    print("Using synthetic wideband spectrum fallback.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
freqs = np.fft.fftshift(np.fft.fftfreq(len(iq), d=1 / fs_iq))
spectrum = np.fft.fftshift(np.fft.fft(iq))
axes[0].plot(freqs, 20 * np.log10(np.abs(spectrum) / len(iq) + 1e-12))
axes[0].set_title("Wideband spectrum")
axes[0].set_xlabel("Frequency offset (Hz)")
axes[0].set_ylabel("Magnitude (dB)")
axes[0].set_ylim(-120, 5)
plot_spectrogram(np.real(iq), fs=fs_iq, ax=axes[1], title="Waterfall-like view", nperseg=2048, noverlap=1536)
axes[1].set_ylim(0, fs_iq / 2)
plt.tight_layout()


## Key Takeaway

An RTL-SDR plus FFTs gives you the core behavior of a spectrum analyzer. The limitation is dynamic range and calibration, not the underlying idea.